# E4 · AfriMGSM — les six états sur l'axe **utilité**

Le modèle résout un problème d'arithmétique et sa réponse est **un nombre**. Comparer des
nombres ne demande de lire aucune langue — donc pas de juge.

**Cet axe n'a jamais servi à l'entraînement**, et c'est précisément son intérêt : il détecte
l'**oubli catastrophique**. Un alignement qui améliorerait la véracité en détruisant
l'arithmétique ne serait pas un progrès, et la littérature signale ce risque
systématiquement.

⚠️ **Aucune contamination ici, contrairement à Uhura.** E4 sur Uhura tombe à 183 questions
sur 808 parce que le DPO s'est entraîné sur le même TruthfulQA. AfriMGSM garde ses **250
questions entières**, et donc toute sa puissance statistique.

**Repère externe** : la carte du modèle annonce 20,79 pour Qwen3.5-4B contre 34,06 pour
AfriqueQwen, mais en **moyenne sur toutes ses langues**. Mesuré ici sur le haoussa seul, A0
donnait 0,1680 et A1 environ 0,28.

### Réglages Kaggle
T4, internet activé, datasets `afrique-safety-dpo-code` et `afrique-safety-dpo-adapters`.
Durée attendue : **~4,4 h** (6 × 44 min). C'est le notebook le plus cher — le seul qui génère
du texte.

## 0 · Environnement

Même préparation que les autres notebooks, et pour les mêmes raisons mesurées : invalidation
implicite via versions épinglées, et `expandable_segments` contre la fragmentation que
provoque un vocabulaire de 248 077 tokens.

Deux assertions refusent de démarrer sur du code périmé — `stopping_criteria` pour la
génération, `per_row` pour la classification. Toutes deux ont été ajoutées après une panne.

In [ ]:
!pip install -q -U "transformers==5.16.1" "peft==0.20.0" bitsandbytes accelerate datasets

In [ ]:
import os, sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


def racine_du_code():
    """Le dossier contenant src/paths.py, ou qu'il soit monte."""
    for base in (Path("/kaggle/input"), Path.cwd(), *Path.cwd().parents):
        if not base.exists():
            continue
        for trouve in base.rglob("paths.py"):
            if trouve.parent.name == "src":
                return trouve.parents[1]
    raise RuntimeError("src/paths.py introuvable. Attacher afrique-safety-dpo-code.")


def chemin_adaptateur(nom):
    """Le dossier nomme `nom` contenant un adapter_config.json.

    Par NOM DE DOSSIER: Kaggle supprime le dossier de tete quand il est seul a la racine du
    zip, donc `adapters/A3_s42_sft/` arrive comme `A3_s42_sft/`.
    """
    for base in (Path("/kaggle/input"), Path.cwd(), *Path.cwd().parents):
        if not base.exists():
            continue
        for trouve in base.rglob("adapter_config.json"):
            if trouve.parent.name == nom:
                return trouve.parent
    raise RuntimeError(f"adaptateur {nom} introuvable. Attacher afrique-safety-dpo-adapters.")


RACINE_CODE = racine_du_code()
sys.path.insert(0, str(RACINE_CODE))

import importlib
import src.eval_mcq, src.eval_tasks
importlib.reload(src.eval_mcq)
importlib.reload(src.eval_tasks)
from src.eval_mcq import mcnemar_p
from src.eval_tasks import evaluate_classification, evaluate_numeric

import inspect
assert "stopping_criteria" in inspect.getsource(evaluate_numeric), "version obsolete"
assert "per_row" in inspect.getsource(evaluate_classification), "version obsolete"

import torch, transformers, peft
print("code :", RACINE_CODE)
print(f"{torch.cuda.get_device_name(0)} | "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} Go")

## 1 · Les six états

Un état est un couple **(backbone, adaptateur)**. L'adaptateur `None` désigne le modèle brut.

**Le garde-fou est essentiel** : un adaptateur posé sur le mauvais backbone produit du bruit
**sans lever la moindre erreur**. Chaque `adapter_config.json` déclare sa base, et on la
compare à celle qu'on charge.

| état | ce qu'il sert à mesurer |
| :---- | :---- |
| A0, A1 | écart **avant** tout alignement |
| A2s, A3s | contribution du **SFT seul** |
| A2d, A3d | **le claim**, après SFT + DPO |

In [ ]:
import gc, json, time

from datasets import load_dataset
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

QWEN = "Qwen/Qwen3.5-4B-Base"
AFRIQUE = "McGill-NLP/AfriqueQwen3.5-4B-50Langs"

ETATS = [
    ("A0_base",       QWEN,    None),
    ("A1_base",       AFRIQUE, None),
    ("A2s_sft",       QWEN,    "A2_s42_sft"),
    ("A3s_sft",       AFRIQUE, "A3_s42_sft"),
    ("A2d_sft_dpo",   QWEN,    "A2_s42_dpo"),
    ("A3d_sft_dpo",   AFRIQUE, "A3_s42_dpo"),
]

CHEMINS = {}
for nom, backbone, adaptateur in ETATS:
    if adaptateur is None:
        CHEMINS[nom] = None
        continue
    chemin = chemin_adaptateur(adaptateur)
    declaree = json.loads((chemin / "adapter_config.json").read_text())["base_model_name_or_path"]
    assert declaree == backbone, f"{adaptateur} entraine sur {declaree}, pas sur {backbone}"
    CHEMINS[nom] = chemin
print(f"{len(ETATS)} etats, toutes les bases declarees concordent")

quant = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
)
SORTIES = Path("/kaggle/working/resultats")
SORTIES.mkdir(parents=True, exist_ok=True)


def charger(nom, backbone):
    tok = AutoTokenizer.from_pretrained(backbone)
    modele = AutoModelForCausalLM.from_pretrained(
        backbone, quantization_config=quant, device_map={"": 0}, dtype=torch.float16
    )
    if CHEMINS[nom] is not None:
        modele = PeftModel.from_pretrained(modele, str(CHEMINS[nom]))
    return tok, modele.eval()


COMPARAISONS = [
    ("ecart de depart", "A0_base",     "A1_base"),
    ("apres SFT seul",  "A2s_sft",     "A3s_sft"),
    ("apres SFT + DPO", "A2d_sft_dpo", "A3d_sft_dpo"),
]


def comparer(justesse, resultats, cle):
    """Les trois ecarts, chacun juge par McNemar puisque les etats voient les memes items."""
    print("=" * 68)
    for etiquette, controle, cible in COMPARAISONS:
        if controle not in justesse or cible not in justesse:
            print(f"{etiquette:<18} incomplet"); continue
        mc = mcnemar_p(justesse[controle], justesse[cible])
        ecart = resultats[cible][cle] - resultats[controle][cle]
        print(f"{etiquette:<18} {ecart:+.4f}   "
              f"desaccords {mc['discordant']:>4}  "
              f"({mc['only_first']} / {mc['only_second']})   p = {mc['p']:.4g}  ->",
              "ECART REEL" if mc["p"] < 0.05 else "indiscernable de zero")

## 2 · Le jeu et l'amorce 8-shot

Les états A0 et A1 sont des checkpoints **base** : sans exemples, ils ne répondent pas, ils
continuent le texte. AfriMGSM fournit huit solutions détaillées **en haoussa** dans son split
`train` — c'est le protocole MGSM standard, et il est appliqué à tous les états pour que la
comparaison reste valide.

**Les chaînes d'arrêt servent deux fois** : interrompre la génération dès la réponse finie, et
tronquer avant le scoring. Sans elles, un modèle base enchaîne sur un exercice **inventé** ;
comme `extract_final_number` prend délibérément le *dernier* nombre, la réponse serait scorée
sur la question hallucinée. Mesuré sur un cas réel : **99 au lieu de 18**.

In [ ]:
amorces = list(load_dataset("masakhane/afrimgsm", "hau", split="train"))
mgsm = list(load_dataset("masakhane/afrimgsm", "hau", split="test"))

prefixe = "".join(f"Tambaya: {e['question']}\nAmsa: {e['answer']}\n\n" for e in amorces)
# Echapper les accolades: le prefixe traverse str.format.
GABARIT = prefixe.replace("{", "{{").replace("}", "}}") + "Tambaya: {question}\nAmsa:"
ARRETS = ["\n\n", "Tambaya:"]
TOKENS_MAX = 128
TOLERANCE = 1e-6

print(f"{len(amorces)} exemples d'amorce | {len(mgsm)} questions de test")
print(f"amorce : {len(prefixe)} caracteres")

### Évaluation

Par blocs de 50 : la génération est lente et, sans découpage, rien ne s'afficherait avant la
fin. **La justesse question par question est conservée** — sans elle, aucun test apparié.

À lire aussi : le taux **« sans nombre »**. Un modèle qui divague sans jamais produire de
nombre échoue autrement qu'un modèle qui calcule mal, et confondre les deux masquerait lequel.

In [ ]:
e4, justesse = {}, {}

for nom, backbone, _ in ETATS:
    if nom in e4:
        continue
    print(f"--- {nom} : chargement", flush=True)
    t0 = time.time()
    tok, modele = charger(nom, backbone)

    justes = sans_nombre = vus = 0
    justesse[nom] = []
    for debut in range(0, len(mgsm), 50):
        out = evaluate_numeric(modele, tok, mgsm[debut:debut + 50], template=GABARIT,
                               max_new_tokens=TOKENS_MAX, stop=ARRETS)
        justes += out["correct"]
        sans_nombre += out["unparsed"]
        vus += out["n"]
        justesse[nom].extend(
            r["predit"] is not None and abs(r["predit"] - r["attendu"]) <= TOLERANCE
            for r in out["records"]
        )
        print(f"   {vus:>3}/{len(mgsm)}  exactitude {justes/vus:.3f}  "
              f"[{(time.time()-t0)/60:.1f} min]", flush=True)

    e4[nom] = {"n": vus, "correct": justes, "accuracy": justes / vus,
               "unparsed": sans_nombre, "unparsed_rate": sans_nombre / vus,
               "min": round((time.time() - t0) / 60, 1)}
    (SORTIES / "E4_afrimgsm.json").write_text(
        json.dumps({"jeu": "afrimgsm hau", "amorce": "8-shot", "n": len(mgsm),
                    "resultats": e4, "justesse": justesse},
                   indent=2, ensure_ascii=False), encoding="utf-8")

    print(f"{nom:<14} {e4[nom]['accuracy']:.4f}   "
          f"sans nombre {e4[nom]['unparsed_rate']:.2f}   [{e4[nom]['min']} min]", flush=True)
    del modele, tok
    gc.collect(); torch.cuda.empty_cache()

## 3 · Les trois écarts

⚠️ **La lecture n'est pas la même que sur Uhura.** Ici on n'entraîne pas : un **gain** n'est
pas attendu, et une **perte** serait le signal d'oubli catastrophique. Ce qui compte est de
savoir si l'alignement a coûté de l'arithmétique, et s'il en a coûté **autant aux deux bras**.

In [ ]:
import pandas as pd

tableau = pd.DataFrame(e4).T[["n", "correct", "accuracy", "unparsed_rate", "min"]]
tableau.columns = ["n", "justes", "exactitude", "sans nombre", "min"]
display(tableau.round(4))

comparer(justesse, e4, "accuracy")

print()
for etiquette, avant, apres in [("A2 : base -> SFT+DPO", "A0_base", "A2d_sft_dpo"),
                                ("A3 : base -> SFT+DPO", "A1_base", "A3d_sft_dpo")]:
    if avant in e4 and apres in e4:
        d = e4[apres]["accuracy"] - e4[avant]["accuracy"]
        mc = mcnemar_p(justesse[avant], justesse[apres])
        print(f"{etiquette:<24} {d:+.4f}   p = {mc['p']:.4g}  ->",
              "PERTE REELLE" if d < 0 and mc["p"] < 0.05 else
              "gain reel" if d > 0 and mc["p"] < 0.05 else "indiscernable")
print("\nUne perte significative ici serait de l'oubli catastrophique: l'alignement aurait")
print("achete de la veracite en detruisant l'arithmetique. Ce n'est pas un progres.")